# Module 11 — Code Along: Building AI Agents with Pydantic AI

Three sections, each building on the last:

1. **Hello world** — the simplest agent, no tools, `TestModel` (no API key)
2. **Your first tool** — `@agent.tool_plain`, then `RunContext` for dependency injection
3. **The catalog agent** — multiple tools wrapping a product catalog

# Section 1 · Hello World with Pydantic AI

The simplest possible agent — no tools, no dependencies, no API key. Just `Agent` + `run_sync`.

In [ ]:
# 1a · The simplest agent — 4 lines, no tools, no API key
from pydantic_ai import Agent

agent = Agent('test', instructions='Be concise, reply with one sentence.')

result = agent.run_sync('Where does "hello world" come from?')
print(result.output)
# 'test' is a fake model built into Pydantic AI — no API key needed.
# It returns a canned response, but proves the wiring works.

In [ ]:
# 1b · With a real model — swap the string, nothing else changes
#
# Pick your provider (set the env var before running):
#   Gemini (free):  export GOOGLE_API_KEY=AIza…   → model='google:gemini-2.5-flash'
#   OpenAI (paid):  export OPENAI_API_KEY=sk-…    → model='openai:gpt-4o-mini'
import os

if os.getenv('GOOGLE_API_KEY'):
    model = 'google:gemini-2.5-flash'
elif os.getenv('OPENAI_API_KEY'):
    model = 'openai:gpt-4o-mini'
else:
    model = None

if model:
    agent_live = Agent(model, instructions='Be concise, reply with one sentence.')
    result = agent_live.run_sync('Where does "hello world" come from?')
    print(f'[{model}]', result.output)
else:
    print('Set GOOGLE_API_KEY or OPENAI_API_KEY to try a real model.')
    print('TestModel demo above proves the wiring works without any API key.')

# Section 2 · Your First Tool

A **tool** is just a Python function the LLM can call. `@agent.tool_plain` is for tools that don't need shared state. `@agent.tool` is for tools that need dependency injection via `RunContext`.

This is the **third decorator moment**:

| Day | Decorator | Framework does |
|---|---|---|
| 1 | `@app.get("/products")` | serves it as an HTTP route |
| 3 | `@pytest.fixture` | injects it as test setup |
| 4 | `@agent.tool` | exposes it as a tool the LLM can call |

In [ ]:
# 2a · A simple tool — @agent.tool_plain (no dependencies needed)
from pydantic_ai import Agent

weather_agent = Agent(
    'test',
    instructions='Use the tools to answer questions about the weather.',
)

@weather_agent.tool_plain
def get_temperature(city: str) -> str:
    """Get the current temperature for a city."""
    # In real life, this would call a weather API
    temps = {'Mumbai': '32°C', 'London': '18°C', 'New York': '25°C'}
    return temps.get(city, f'No data for {city}')

result = weather_agent.run_sync('What is the temperature in Mumbai?')
print('Output:', result.output)

# TestModel calls every registered tool with default args.
# The point: @agent.tool_plain registered the function, and the agent used it.

In [ ]:
# 2b · RunContext — tools that need shared data (dependency injection)
from pydantic_ai import Agent, RunContext

# deps_type declares what type tools receive via ctx.deps
inventory_agent = Agent(
    'test',
    deps_type=list,
    instructions='You help answer questions about inventory items. Use tools.',
)

@inventory_agent.tool
def count_items(ctx: RunContext[list]) -> int:
    """Count how many items are in the inventory."""
    return len(ctx.deps)

@inventory_agent.tool
def find_item(ctx: RunContext[list], name: str) -> str:
    """Check if an item exists in the inventory."""
    matches = [item for item in ctx.deps if name.lower() in item.lower()]
    return f"Found: {matches}" if matches else f"'{name}' not found"

# deps= passes your data to every tool via ctx.deps
items = ['Wireless Mouse', 'Mechanical Keyboard', 'USB-C Hub']
result = inventory_agent.run_sync('How many items do we have?', deps=items)
print('Output:', result.output)

# Same pattern as Day 3: inject the dependency, swap it in tests.

## `@agent.tool` vs `@agent.tool_plain`

| | `@agent.tool` | `@agent.tool_plain` |
|---|---|---|
| First parameter | `ctx: RunContext[T]` | *your* params |
| Access to deps | `ctx.deps` | none |
| Use when | tools need shared state (a catalog, a DB, a client) | tools are self-contained (math, formatting) |

# Section 3 · Multiple Tools — the Catalog Agent

Build the same pattern you'll use in the lab: multiple tools wrapping a data source, injected via `RunContext`. This is a miniature version of the `CatalogAgent` you'll build in Lab 11.

In [ ]:
# 3a · Define product data + a catalog agent with three tools
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class Product:
    id: int
    name: str
    price: float
    category: str
    in_stock: bool = True

PRODUCTS = [
    Product(1, 'Wireless Mouse', 899.0, 'Electronics'),
    Product(2, 'Mechanical Keyboard', 5499.0, 'Electronics'),
    Product(3, 'Yoga Mat', 1299.0, 'Fitness'),
    Product(4, 'Running Shoes', 3999.0, 'Fitness', in_stock=False),
    Product(5, 'USB-C Hub', 1999.0, 'Electronics'),
]

catalog_agent = Agent(
    'test',
    deps_type=list,
    instructions=(
        'You are a product catalog assistant. '
        'Use the tools to answer questions about products. '
        'Never guess — always look up the data first.'
    ),
)

@catalog_agent.tool
def list_all_products(ctx: RunContext[list]) -> list[dict]:
    """Return every product in the catalog."""
    return [{'id': p.id, 'name': p.name, 'price': p.price,
             'category': p.category, 'in_stock': p.in_stock}
            for p in ctx.deps]

@catalog_agent.tool
def search_by_name(ctx: RunContext[list], term: str) -> list[dict]:
    """Find products whose name contains the search term (case-insensitive)."""
    return [{'id': p.id, 'name': p.name, 'price': p.price}
            for p in ctx.deps if term.lower() in p.name.lower()]

@catalog_agent.tool
def count_by_category(ctx: RunContext[list]) -> dict[str, int]:
    """Count how many products are in each category."""
    counts: dict[str, int] = {}
    for p in ctx.deps:
        counts[p.category] = counts.get(p.category, 0) + 1
    return counts

print(f'Catalog agent defined — {len(PRODUCTS)} products, 3 tools registered')

In [ ]:
# 3b · Run with TestModel — proves the wiring works, no API key needed
result = catalog_agent.run_sync(
    "What's our most expensive product?",
    deps=PRODUCTS,
)
print('Output:', result.output)
print('\nTestModel called every tool with default args — smoke test passed.')
print('In the lab, you build this with your real ProductCatalog class.')

In [ ]:
# 3c · (OPTIONAL) Run with a real LLM — swap one string
import os

if os.getenv('GOOGLE_API_KEY'):
    model = 'google:gemini-2.5-flash'
elif os.getenv('OPENAI_API_KEY'):
    model = 'openai:gpt-4o-mini'
else:
    model = None

if model:
    result = catalog_agent.run_sync(
        "What's our most expensive product?",
        deps=PRODUCTS,
        model=model,
    )
    print(f'[{model}]', result.output)
else:
    print('Set GOOGLE_API_KEY or OPENAI_API_KEY to run live.')
    print('TestModel demo above proves the wiring works.')

## What you just built

| Piece | What it does |
|---|---|
| `Agent(deps_type=..., instructions=...)` | Creates the agent — declares what tools receive and how the LLM behaves |
| `@agent.tool` / `@agent.tool_plain` | Registers a Python function as a tool the LLM can call |
| `RunContext[T]` | Injects dependencies into tools — same DI pattern as Day 3 |
| `agent.run_sync(prompt, deps=...)` | Runs the agent loop: plan → act → observe → answer |
| `model='test'` | Fake LLM for testing — no API key needed |
| `model='google:gemini-2.5-flash'` | Gemini (free tier — `GOOGLE_API_KEY`) |
| `model='openai:gpt-4o-mini'` | OpenAI (paid — `OPENAI_API_KEY`) |

**→ Lab 11** builds the real `CatalogAgent` with your `ProductCatalog` class and four `@agent.tool` functions.